# Direct Preference Optimization (DPO)

С этого короткого семинара мы начнем рабираться еще один важный (может быть даже самый важный) этап пост-обучения языковых моделей - обучение на фидбеке/предпочтениях. Эту стадию еще называют alignment, но кажется в последнее время это не очень распространенный термин. 

Дообучение на инструкциях позволяет перестроить модель на нужный формат, который удобно использовать на практике. Intrust модель часто уже может правильно отвечать на вопросы, но помимо просто правильности есть еще множетсво других аспектов, которые характеризуются полезность модели (безопасность, отзывчивость, многословность, умение сказать нет, и т.д.) Собирать правильные ответы по такие критерии очень тяжело. Часто даже сформулировать их в четкой форме бывает сложно, но почти всегда можно оценить какой из двух вариантов лучше. В этом и состоит основное отличие этого этапа обучения - вместо обучения на паре промпт-правильный ответ, мы обучаемся на тройках промпт-ответ-оценка. В процессе обучения модель обучается генерировать ответы, которые получают лучшие оценки. 

И такой подход можно масштабировать еще дальше. Какое-то количество оценок нам нужно разметить вручную, но потом мы можем обучить модель, которая будет предсказывать оценку для любого текста и используя эту модель мы можем запустить цикл обучения с подкреплением (reinforcement learning), в котором модель будет генерировать ответ на случайный промпт, а Reward модель предсказывать его оценку. Также для части задача можно написать простые скоринг функции (синтаксическая правильность кода, близость численного ответа правильному и т.д.), которые заменят reward модель. Но про это мы поговорим на следующем занятии (RLHF и GRPO). 

В этом семинаре мы разберём **Direct Preference Optimization (DPO)**, это что-то среднее между RL и SFT. Он основан на стандартном supervised обучении, но позволяет обучаться на предпочтениях также как и RL алгоритмы. DPO позволяет обучать модель **напрямую на данных с предпочтениями** (пары chosen / rejected) без отдельного обучения reward-модели и без RL-цикла. Это делает его гораздо удобнее! (в следующих занятиях мы увидим, что для обучения RLHF мы не можем использовать уже готовые оценочные датасеты, так как важно, чтобы изначальные тексты были сгенерированы изначальной моделью; в DPO такого ограничения нет и мы можем использовать любые готовые датасеты предпочтений для любых моделей)




Сама [статья](https://arxiv.org/abs/2305.18290) про DPO вышла после работ OpenAI про instruct gpt. DPO предлагает упрощение, позволяющую переформулировать задачу как **supervised обучение** прямо на preference-данных.

Ключевая идея: reward model нам больше не нужна в явном виде — она неявно *встроена* в отношение вероятностей обучаемой и reference модели.

### Лосс DPO

$$
\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x, y^+, y^-)} \left[
    \log \sigma \!\left(
        \beta \left(
            \log \frac{\pi_\theta(y^+ \mid x)}{\pi_{\text{ref}}(y^+ \mid x)}
            - \log \frac{\pi_\theta(y^- \mid x)}{\pi_{\text{ref}}(y^- \mid x)}
        \right)
    \right)
\right]
$$

- $\pi_\theta$ — обучаемая модель (policy)
- $\pi_{\text{ref}}$ — reference модель (замороженная копия исходной модели)
- $y^+$ — предпочтительный ответ (chosen)
- $y^-$ — нежелательный ответ (rejected)
- $\beta$ — температурный параметр: чем больше, тем сильнее штрафуем отклонения от reference модели

### Что происходит при обучении

Лосс **максимизирует разрыв** между лог-вероятностями chosen и rejected ответов, *относительно* reference модели. Reference модель - это очень важный параметр во всех таких алгоритмах. Сложность со всеми алгоритмами дообучения в том, что модель может забыть знания, которые она преобрела на предыдущих фазах. Reference модель создает исходную точку для модели, с которой она всегда сравнивается. Таким образом получается, что модель подстраивается генерировать ответы, получающие лучшие скоры, при это минимально возможно отклоняясь от изначального соостояния. 


[Видео с подробным и достаточно понятным разбором DPO](https://www.youtube.com/watch?v=hvGa5Mba4c8) 


В отличие от PPO (RLHF), где нужно сразу четыре модели (!) в DPO нужно только две - одна обучаемая и одна reference. 
DPO также совместимо с PEFT и можно дообучать адаптеры, а не целую модель. Таким образом даже не нужно хранить две версии большой модели в памяти, нужно просто делать предсказания с активным адаптером и без.

Давайте попробуем дообучить какую-нибудь модель на датасете предпочтений с помощью DPO.

In [13]:
# Установка зависимостей

In [1]:
%pip install -q trl transformers datasets peft bitsandbytes accelerate

Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, PeftModel
from trl import DPOTrainer, DPOConfig
from tqdm.notebook import tqdm

pd.set_option('display.max_colwidth', 500)

##  Датасет

Для DPO нужен **preference dataset** с тремя полями:
- `prompt` — входной запрос
- `chosen` — предпочтительный ответ
- `rejected` — нежелательный ответ

(выше я говорил про промпт-ответ-скор, но в какой-то степени это то же самое только тут промпт-ответ-0 и промпт-ответ-1, в таких алгоритмах обычно используют один промпт и из него генерируют несколько разных ответов для сравнения, их может быть 2, а может и больше, но даже если их больше то всегда можно перевести их в попарные chosen-rejected, а из пар получить скоры для каждой комбинации)

Используем [`trl-lib/ultrafeedback_binarized`](https://huggingface.co/datasets/trl-lib/ultrafeedback_binarized) — 62k примеров в формате диалогов, полученных из UltraFeedback.

> Современный TRL понимает **conversational format** автоматически: если в `chosen`/`rejected` лежат списки сообщений `[{"role": ..., "content": ...}]`, трейнер сам применит chat template модели. Вручную форматировать промпты не нужно.

In [2]:
dataset = load_dataset("trl-lib/ultrafeedback_binarized", split="train")
print(f"Размер датасета: {len(dataset)}")
print(f"Колонки: {dataset.column_names}")

Размер датасета: 62135
Колонки: ['chosen', 'rejected', 'score_chosen', 'score_rejected']


In [3]:
# Посмотрим на один пример
example = dataset[0]

print("=== PROMPT (chosen[:-1]) ===")
for msg in example["chosen"][:-1]:
    print(f"[{msg['role']}]: {msg['content'][:300]}")

print("\n=== CHOSEN RESPONSE (chosen[-1]) ===")
msg = example["chosen"][-1]
print(f"[{msg['role']}]: {msg['content'][:300]}")

print("\n=== REJECTED RESPONSE (rejected[-1]) ===")
msg = example["rejected"][-1]
print(f"[{msg['role']}]: {msg['content'][:300]}")

print(f"\nscore_chosen={example['score_chosen']}, score_rejected={example['score_rejected']}")

=== PROMPT (chosen[:-1]) ===
[user]: Use the pygame library to write a version of the classic game Snake, with a unique twist

=== CHOSEN RESPONSE (chosen[-1]) ===
[assistant]: Sure, I'd be happy to help you write a version of the classic game Snake using the pygame library! Here's a basic outline of how we can approach this:

1. First, we'll need to set up the game display and create a game object that we can use to handle the game's state.
2. Next, we'll create the game'

=== REJECTED RESPONSE (rejected[-1]) ===
[assistant]: Sure, here's an example of how to write a version of Snake game with a unique twist using the Pygame library:
```python
import pygame

class SnakeGame:
    def __init__(self, game_width, game_height):
        pygame.init()
        screen = pygame.display.set_mode((game_width, game_height))
        p

score_chosen=6.0, score_rejected=4.0


In [4]:
# Берём небольшой срез для демонстрации
dataset_train = dataset.select(range(10000))
dataset_eval  = dataset.select(range(10000, 10300))

print(f"Train: {len(dataset_train)}, Eval: {len(dataset_eval)}")

Train: 10000, Eval: 300


Загрузка модели (QLoRA)

Используем `Qwen/Qwen2.5-3B-Instruct` — instruction-tuned модель, что типично для DPO: сначала SFT, потом preference alignment.

DPO применяем поверх LoRA-адаптеров: это позволяет обучать только маленький набор параметров и сохранять только адаптер (~150 MB вместо ~6 GB для full fine-tune).

In [5]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

# Квантизация в 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False  # необходимо для gradient checkpointing

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

# DPOTrainer ожидает padding слева (важно для батчей при генерации)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

### LoRA-конфиг

Добавляем адаптеры ко всем линейным слоям. `target_modules="all-linear"` — рекомендованный вариант в современном PEFT.

Основные параметры LoRA:
- `r` — ранг адаптера (меньше → меньше параметров и памяти, но меньше выразительности)
- `lora_alpha` — масштаб адаптера (обычно равен или в 2× больше `r`)
- `lora_dropout` — дропаут внутри адаптера

In [7]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

##  Обучение

### DPOConfig

Ключевые параметры:

| Параметр | Значение | Описание |
|---|---|---|
| `beta` | 0.1 | Сила KL-регуляризации. Чем выше — тем ближе к reference модели |
| `loss_type` | `"sigmoid"` | Стандартный DPO (Bradley-Terry). Альтернативы: `"ipo"`, `"hinge"`, `"robust"` |
| `max_prompt_length` | 512 | Максимальная длина промпта |
| `max_length` | 1024 | Максимальная суммарная длина (prompt + response) |
| `learning_rate` | 5e-5 | Для LoRA можно взять выше, чем для full fine-tune |
| `gradient_checkpointing` | `True` | Экономит память за счёт пересчёта активаций |

> **Reference model**: если не передавать `ref_model` явно, DPOTrainer автоматически использует начальное состояние policy модели в качестве reference. При использовании PEFT это работает через отдельную копию весов.

> **Метрики**, за которыми стоит следить во время обучения:
> - `rewards/accuracies` — доля примеров, где chosen > rejected (цель → 1.0)
> - `rewards/margins` — разрыв между chosen и rejected наградами (цель → расти)
> - `logps/chosen` и `logps/rejected` — лог-вероятности ответов

In [8]:
training_args = DPOConfig(
    output_dir="dpo_qwen_output",
    # DPO-specific
    beta=0.1,
    loss_type="sigmoid",  # стандартный DPO
    max_length=1024,
    # Обучение
    num_train_epochs=1,
    max_steps=1000,          # для демонстрации; убрать для полного обучения
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    # Сохранение и логирование
    save_strategy="no",
    logging_steps=10,
    report_to="none",       # заменить на "wandb" для трекинга
    # Precision
    bf16=True,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [9]:
dpo_trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_eval,
    processing_class=tokenizer,
    peft_config=peft_config,
    # ref_model=None  # автоматически использует начальное состояние model
)

In [10]:
# Запуск обучения
dpo_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.687705
20,0.704330
30,0.690078
40,0.667506
50,0.652461
60,0.639644
70,0.587980
80,0.528310
90,0.614989
100,0.508083


KeyboardInterrupt: 

In [11]:
# Сохраняем только LoRA-адаптер 
dpo_trainer.model.save_pretrained("dpo_qwen_adapter")
tokenizer.save_pretrained("dpo_qwen_adapter")
print("Адаптер сохранён в ./dpo_qwen_adapter")

Адаптер сохранён в ./dpo_qwen_adapter


## Сравнение моделей до и после DPO

Загрузим базовую модель и модель с DPO-адаптером, сгенерируем ответы на одни и те же промпты и сравним результаты.

In [8]:
# Базовая модель (без адаптера)
model_base = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model_base.eval()

tokenizer_eval = AutoTokenizer.from_pretrained(model_name)
tokenizer_eval.padding_side = "left"
if tokenizer_eval.pad_token is None:
    tokenizer_eval.pad_token = tokenizer_eval.eos_token

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [9]:
def generate_responses(model, prompts, tokenizer, max_new_tokens=300):
    """Генерируем ответы батчами."""
    responses = []
    batch_size = 2
    device = next(model.parameters()).device

    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i + batch_size]

        # Применяем chat template к каждому промпту
        formatted = [
            tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True)
            for p in batch_prompts
        ]

        inputs = tokenizer(
            formatted,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        ).to(device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        # Обрезаем входные токены, оставляем только сгенерированные
        generated = output_ids[:, inputs["input_ids"].shape[-1]:]
        responses.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))

    return responses

In [10]:
# Берём 50 тестовых примеров
n_eval = 50
eval_subset = dataset_eval.select(range(n_eval))

# Промпт = все сообщения chosen, кроме последнего ответа ассистента
prompts = [example[:-1] for example in eval_subset["chosen"]]

print("Генерируем ответы базовой модели...")
responses_base = generate_responses(model_base, prompts, tokenizer_eval)

Генерируем ответы базовой модели...


  0%|          | 0/25 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [11]:
# Модель с DPO-адаптером
model_dpo = PeftModel.from_pretrained(model_base, "dpo_qwen_adapter")
model_dpo.eval()

print("Генерируем ответы DPO модели...")
responses_dpo  = generate_responses(model_dpo,  prompts, tokenizer_eval)

Генерируем ответы DPO модели...


  0%|          | 0/25 [00:00<?, ?it/s]

In [16]:
# код для интерактивного сравнения предсказаний (но результаты не сохраняются)

In [13]:
import ipywidgets as widgets
from IPython.display import display, HTML

def show_example(i):
    prompt_text = "\n".join(f"[{m['role']}] {m['content']}" for m in prompts[i])
    chosen_text = eval_subset["chosen"][i][-1]["content"]
    base_text   = responses_base[i]
    dpo_text    = responses_dpo[i]

    def box(label, text, color):
        return f"""
        <div style="margin-bottom:8px">
          <b style="color:{color}">{label}</b>
          <div style="background:#f8f8f8; border-left:4px solid {color};
                      padding:8px 12px; white-space:pre-wrap; font-size:13px">{text}</div>
        </div>"""

    return f"""
    <div style="font-family:sans-serif; max-width:900px; margin:10px 0">
      <h4 style="margin-bottom:6px">Example {i+1} / {len(prompts)}</h4>
      {box("Prompt", prompt_text, "#555")}
      {box("Chosen (reference answer)", chosen_text, "#2a7")}
      {box("Base model", base_text, "#e88")}
      {box("DPO model", dpo_text, "#48c")}
    </div>"""

slider  = widgets.IntSlider(value=0, min=0, max=len(prompts)-1, step=1,
                            description="Example:", continuous_update=False,
                            layout=widgets.Layout(width="400px"))
btn_prev = widgets.Button(description="◀ Prev", layout=widgets.Layout(width="90px"))
btn_next = widgets.Button(description="Next ▶", layout=widgets.Layout(width="90px"))
out      = widgets.Output()

def render():
    out.clear_output(wait=True)
    with out:
        display(HTML(show_example(slider.value)))

def on_prev(_):
    if slider.value > 0:
        slider.value -= 1

def on_next(_):
    if slider.value < slider.max:
        slider.value += 1

btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
slider.observe(lambda _: render(), names="value")

display(widgets.HBox([btn_prev, slider, btn_next]), out)
render()

Output()

In [14]:
# Поменяйте индекс чтобы посмотреть другой пример
i = 0

prompt_text = "\n".join(f"[{m['role']}] {m['content']}" for m in prompts[i])
chosen_text = eval_subset["chosen"][i][-1]["content"]

print(f"=== Example {i+1} ===")
print(f"\n--- PROMPT ---\n{prompt_text}")
print(f"\n--- CHOSEN ---\n{chosen_text}")
print(f"\n--- BASE MODEL ---\n{responses_base[i]}")
print(f"\n--- DPO MODEL ---\n{responses_dpo[i]}")

=== Example 1 ===

--- PROMPT ---
[user] Using the given customer requirements, analyze the following data set to recommend a suitable laptop. The data set is in an Excel table format and contains the following columns: Laptop Model, Screen Size, RAM Size, Storage Capacity, Graphics Card, Price. Please provide a SQL query that filters the data set to only include laptops that meet the customer's requirements and sorts them by price in ascending order. Additionally, please provide a Python code snippet that calculates the average price of the recommended laptops.

--- CHOSEN ---
First, let's convert the customer requirements into a SQL query. I don't have the exact requirements, so I have used placeholders that you can replace with the actual values. Then, I'll provide a Python code snippet to calculate the average price of the recommended laptops.

Assuming you have the following customer requirements:
1. Minimum screen size (replace with 'min_screen_size')
2. Minimum RAM size (replace

### На что смотреть в результатах

Разница может быть не всегда очевидна визуально — preference feedback может быть очень абстрактным и оценивать по многим критериям сразу. Проще всего прочитать ответы до и после и постараться оценить их самостоятельно. Или же посчитать какие-то простые показательные статистики (длина ответа, использование форматирования текста, эмоджи и тп)

В примере выше видно, что предсказание стало длинее и чуть более структурировано

## Варианты и расширения DPO

### Другие loss_type в TRL

В TRL есть несколько вариантов лосса:

| `loss_type` | Описание |
|---|---|
| `"sigmoid"` | Стандартный DPO (Bradley-Terry) |
| `"ipo"` | IPO — избегает переобучения на логит-трансформе |
| `"hinge"` | RSO / SLiC — hinge loss на нормализованных вероятностях |
| `"robust"` | Robust DPO — устойчив к зашумлённым предпочтениям |
| `"bco_pair"` | BCO — обучает бинарный классификатор |

### Комбинирование лоссов (MPO)

Можно комбинировать несколько лоссов:

```python
training_args = DPOConfig(
    loss_type=["sigmoid", "bco_pair", "sft"],
    loss_weights=[0.8, 0.2, 1.0]
)
```

### Итеративный DPO

Вместо статичного датасета: обучаем модель → генерируем новые ответы → собираем предпочтения → повторяем. Так работают более свежие методы типа **Online DPO**.
